In [2]:
import io
import zipfile
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pypdf import PdfReader

# Reproducibility: one documented seed (see AGENTS.md, "Scientific constraints").
SEED = 0
random.seed(SEED)
np.random.seed(SEED)

In [4]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
FV_ZIP = DATA_DIR / 'MMCBNU_6000.zip'

assert FV_ZIP.exists(), f'FingerVein archive not found: {FV_ZIP} (see "Preliminary operations")'
print('Repository root :', PROJECT_ROOT)
print('FingerVein zip  :', FV_ZIP.name, f'({FV_ZIP.stat().st_size / 1e6:.0f} MB)')

Repository root : /Users/jiahaoguo/Desktop/deeplearning/PPG_FingerVein_age_detection
FingerVein zip  : MMCBNU_6000.zip (677 MB)


In [ ]:

def load_subject_info(zip_path):
    """Parse the 100-row 'Information table' from the description PDF inside the archive."""
    with zipfile.ZipFile(zip_path) as z:
        pdf_name = next(n for n in z.namelist() if n.lower().endswith('.pdf'))
        reader = PdfReader(io.BytesIO(z.read(pdf_name)))
    text = '\n'.join((page.extract_text() or '') for page in reader.pages)
    rows = []
    for line in text.splitlines():
        toks = line.split()
        # a data row starts with: <order:int> <age:int> ...
        if len(toks) >= 4 and toks[0].isdigit() and toks[1].isdigit():
            gi = next((i for i, t in enumerate(toks[2:], 2) if t in ('M', 'F')), None)
            if gi is None:
                continue
            rows.append({
                'Order': int(toks[0]),
                'Subject': f'{int(toks[0]):03d}',          # matches the image folders
                'Age': int(toks[1]),
                'Nationality': ' '.join(toks[2:gi]),
                'Gender': toks[gi],
                'Blood type': ' '.join(toks[gi + 1:]) or None,
            })
    return pd.DataFrame(rows).sort_values('Order').reset_index(drop=True)


info = load_subject_info(FV_ZIP)
info.head()

missing = set(range(1, 101)) - set(info['Order'])
assert not missing, f"Soggetti mancanti: {sorted(missing)}"

LABELS_CSV = "subject_age_labels.csv"
info[['Subject', 'Age', 'Gender']].to_csv(LABELS_CSV, index=False)
print(f"Salvato {LABELS_CSV} con {len(info)} righe")

#age_by_subject = dict(zip(info['Subject'], info['Age']))

subjects: 100
Salvato subject_age_labels.csv con 100 righe
